In [1]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
from google.colab import files

uploaded = files.upload()

Saving train_reviews_clean.csv to train_reviews_clean.csv


In [3]:
import pandas as pd

df = pd.read_csv("train_reviews_clean.csv")

print(df.shape)
df.head()

(25000, 6)


,review,label,review_length,sentiment,clean_review,clean_length
0,Bromwell High is a cartoon comedy. It ran at t...,1,806,Positive,Bromwell High is a cartoon comedy. It ran at t...,806
1,Homelessness (or Houselessness as George Carli...,1,2366,Positive,Homelessness (or Houselessness as George Carli...,2322
2,Brilliant over-acting by Lesley Ann Warren. Be...,1,841,Positive,Brilliant over-acting by Lesley Ann Warren. Be...,841
3,This is easily the most underrated film inn th...,1,663,Positive,This is easily the most underrated film inn th...,663
4,This is not the typical Mel Brooks film. It wa...,1,647,Positive,This is not the typical Mel Brooks film. It wa...,647


In [4]:
print(df.shape)

(25000, 6)


In [5]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [7]:
df = pd.read_csv("train_reviews_clean.csv")

print(df.shape)

(25000, 6)


In [8]:
X = df["clean_review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(len(X_train))
print(len(X_test))

20000
5000


In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenization Complete")

Tokenization Complete


In [11]:
class IMDbDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

In [12]:
train_dataset = IMDbDataset(
    train_encodings,
    y_train
)

test_dataset = IMDbDataset(
    test_encodings,
    y_test
)

print(len(train_dataset))
print(len(test_dataset))

20000
5000


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model Loaded")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded


In [14]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        )
    }

In [15]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=2,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

In [16]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.260428,0.260627,0.897400
2,0.131263,0.310471,0.909200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=0.21868478927612306, metrics={'train_runtime': 1002.5208, 'train_samples_per_second': 39.899, 'train_steps_per_second': 2.494, 'total_flos': 2649347973120000.0, 'train_loss': 0.21868478927612306, 'epoch': 2.0})

In [18]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.131263,0.260627,2,0.897400


{'eval_loss': 0.2606273889541626, 'eval_accuracy': 0.8974}


In [19]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.93      0.86      0.89      2500
           1       0.87      0.94      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



In [1]:
trainer.save_model("distilbert_imdb")
tokenizer.save_pretrained("distilbert_imdb")

NameError: name 'trainer' is not defined